# 01 - Extraction des caracteristiques

Ce notebook transforme chaque image du corpus en un vecteur de caracteristiques, celui qu'utilise la steganalyse classique. Ces vecteurs serviront ensuite a entrainer et tester les detecteurs.

Par defaut on utilise SPAM, un descripteur classique leger (686 dimensions). Il est bien plus rapide que SRM complet, qui prend des dizaines d'heures sur ce corpus, tout en restant suffisant pour demontrer les phenomenes etudies. SRM reste disponible en reference sur un petit sous-ensemble.

Chaque resultat est mis en cache sur Drive, et le calcul reprend la ou il s'est arrete si la session se coupe.

## Parametres, Drive et corpus

In [ ]:
import os, glob, shutil
import numpy as np

SEED = 42
np.random.seed(SEED)

ROOT = '/content/corpus'
ALGOS = ['lsb', 'uniward', 'hill']
PAYLOADS = [0.2, 0.4]
SOURCES = ['natural', 'sd', 'sdxl', 'adm']

# Type de caracteristiques. 'spam' est rapide et suffisant, c'est le choix par defaut.
# 'srmq1' est une variante SRM allegee, 'srm' est la reference complete mais tres lente.
FEATURE = 'spam'

In [ ]:
# On monte Drive, puis on restaure le corpus si la session est neuve
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/memoire_data'
except Exception as e:
    DRIVE = '/content/memoire_data'
    print('Drive non monte, sauvegarde locale.', e)
os.makedirs(DRIVE, exist_ok=True)

# Dossier de cache, separe par type de caracteristiques
FEAT_DIR = f'{DRIVE}/features_{FEATURE}'
os.makedirs(FEAT_DIR, exist_ok=True)

# Restauration du corpus depuis la derniere archive datee
if not os.path.isdir(f'{ROOT}/natural/cover'):
    zips = sorted(glob.glob(f'{DRIVE}/corpus_*.zip'))
    if zips:
        print('Restauration du corpus depuis', zips[-1])
        shutil.unpack_archive(zips[-1], ROOT)
    else:
        print('Aucun corpus trouve. Executez d\'abord le notebook 00.')
print('Cache des caracteristiques :', FEAT_DIR)

## Installation

In [ ]:
!pip install -q sealwatch imageio scipy
print('Installation terminee.')

## 1. Fonction d'extraction

SPAM modelise les differences entre pixels voisins par une chaine de Markov. SRM applique une batterie de filtres passe haut. Les deux prennent un tableau, pas un chemin, donc on charge l'image avant.

In [ ]:
import imageio.v2 as imageio
import sealwatch as sw

def load_gray(path):
    x = np.asarray(imageio.imread(path))
    if x.ndim == 3:
        x = x[..., 0]
    return x

def to_vec(f):
    # La sortie sealwatch peut etre un dict de sous modeles, on l'aplatit
    if isinstance(f, dict):
        return np.asarray(sw.tools.flatten(f), dtype=np.float32).ravel()
    return np.asarray(f, dtype=np.float32).ravel()

def choisir_extracteur():
    if FEATURE == 'spam':
        return sw.spam
    if FEATURE == 'srmq1':
        return sw.srmq1
    return sw.srm

def extract(path):
    return to_vec(choisir_extracteur().extract(load_gray(path)))

# Petit test sur une image pour connaitre la dimension et le temps
import time
un = sorted(glob.glob(f'{ROOT}/natural/cover/*.pgm'))[:1]
if un:
    t = time.time()
    d = extract(un[0]).shape[0]
    print(f'dimension : {d}, temps par image : {time.time()-t:.3f}s')

## 2. Extraction avec cache reprenable

Pour chaque ensemble d'images, on sauvegarde un fichier de caracteristiques. Si le fichier final existe deja, on le recharge sans recalculer. Si un calcul a ete interrompu, on reprend au dernier point sauvegarde.

In [ ]:
def extract_set(paths, cache_file, pas=100):
    # Renvoie les caracteristiques d'un ensemble, en reprenant un calcul interrompu
    if os.path.exists(cache_file):
        return np.load(cache_file)                 # deja termine
    tmp = cache_file + '.part.npy'
    feats = []
    if os.path.exists(tmp):
        feats = list(np.load(tmp))
        print('   reprise a', len(feats), 'images')
    for i in range(len(feats), len(paths)):
        feats.append(extract(paths[i]))
        if (i + 1) % pas == 0:
            np.save(tmp, np.array(feats, dtype=np.float32))   # point de reprise
            print('   ', i + 1, '/', len(paths))
    arr = np.array(feats, dtype=np.float32)
    np.save(cache_file, arr)
    if os.path.exists(tmp):
        os.remove(tmp)
    return arr

def paths_of(source, setname):
    # setname vaut 'cover' ou '<algo>_p<payload>'
    if setname == 'cover':
        return sorted(glob.glob(f'{ROOT}/{source}/cover/*.pgm'))
    algo, p = setname.split('_p')
    return sorted(glob.glob(f'{ROOT}/{source}/{algo}/*_p{p}.pgm'))

print('Fonctions de cache pretes.')

## 3. Lancer l'extraction sur tout le corpus

On parcourt chaque source et chaque ensemble. Vous pouvez relancer cette cellule autant de fois que necessaire, elle ne recalcule que ce qui manque.

In [ ]:
setnames = ['cover'] + [f'{a}_p{p}' for a in ALGOS for p in PAYLOADS]

for source in SOURCES:
    for setname in setnames:
        paths = paths_of(source, setname)
        if not paths:
            print(f'{source} {setname}: aucune image, ignore')
            continue
        cache_file = f'{FEAT_DIR}/{source}__{setname}.npy'
        if os.path.exists(cache_file):
            print(f'{source} {setname}: deja en cache')
            continue
        print(f'{source} {setname}: extraction de {len(paths)} images')
        extract_set(paths, cache_file)
print('\nExtraction terminee ou reprise. Relancez si une coupure a interrompu.')

## 4. Verification

On liste les caches produits et on verifie leurs dimensions, pour s'assurer que tout est coherent avant les experiences.

In [ ]:
caches = sorted(glob.glob(f'{FEAT_DIR}/*.npy'))
print(f'{len(caches)} fichiers de caracteristiques dans {FEAT_DIR}\n')
for c in caches:
    arr = np.load(c, mmap_mode='r')
    print(f'  {os.path.basename(c):22s} {arr.shape}')

attendu = len(SOURCES) * len(setnames)
print(f'\n{len(caches)} caches sur {attendu} attendus.')

## Suite

Les caracteristiques sont en cache sur Drive. Le notebook suivant, `02_experience_A_domain_shift`, les charge directement pour entrainer les detecteurs et mesurer le decalage de domaine.

Pensez a enregistrer ce notebook sur GitHub et a noter l'etape dans le README.